# ARCHS4 tissue feature importance (binary RF + SHAP)

For each tissue, train a binary RF classifier (tissue vs. rest) using keyword-matched
sample labels from the ARCHS4 h5 metadata.  
SHAP values identify which LVs most distinguish each tissue.

💡 **Environment:** `clamp-analyses`

In [ ]:
import gc

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from pyprojroot.here import here

import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri

readRDS = ro.r["readRDS"]

np.random.seed(42)

## Settings

In [ ]:
GLOBAL_SEED      = 42
MIN_SAMPLES      = 500       # skip tissues with fewer positive samples
SC_PROB_THRESH   = 0.5      # exclude single-cell samples
MIN_RF_ACCURACY  = 0.9      # minimum balanced accuracy to keep a tissue

PARAM_GRID = {
    'n_estimators':      [100, 300, 500, 1000],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
}

## Output

In [ ]:
OUTPUT_DIR = here('output/archs4_feature_importance_binary_rf')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## Load ARCHS4 model

In [ ]:
def extract_B_matrix(rds_obj):
    B_matrix = rds_obj.rx2("B")
    with localconverter(ro.default_converter + pandas2ri.converter):
        B_values = ro.conversion.rpy2py(B_matrix)
    return pd.DataFrame(
        data=B_values,
        index=B_matrix.rownames if B_matrix.rownames else None,
        columns=B_matrix.colnames if B_matrix.colnames else None,
    )


model_rds = readRDS(str(here("output/archs4/c2cp_coverage_rs100_seed_1/CLAMPfull_C2CP.rds")))

# B matrix rows = LVs, cols = samples → transpose to samples × LVs
B_df      = extract_B_matrix(model_rds)
lv_matrix = B_df.T

model_rds = None
B_df = None
gc.collect()

print(f"lv_matrix: {lv_matrix.shape}  (samples × LVs)")
lv_matrix.head()

## Load ARCHS4 metadata

In [ ]:
h5_path = str(here("data/archs4/human_gene_v2.5.h5"))

def _decode(arr):
    return np.array([x.decode("utf-8", errors="replace") for x in arr])

with h5py.File(h5_path, "r") as f:
    geo_acc = _decode(f["meta/samples/geo_accession"][:])
    source  = _decode(f["meta/samples/source_name_ch1"][:])
    char    = _decode(f["meta/samples/characteristics_ch1"][:])
    sc_prob = f["meta/samples/singlecellprobability"][:]

print(f"Metadata loaded: {len(geo_acc):,} samples")

## Tissues

Same tissue–keyword pairs as `02_biology_validation.ipynb`.

In [ ]:
TISSUE_KEYWORDS = {
    "Adipose":        "adipose",
    "Adrenal gland":  "adrenal gland",
    "Artery":         "artery",
    "Bladder":        "bladder",
    "Blood vessel":   "blood vessel",
    "Brain":          "brain",
    "Breast":         "breast",
    "Cervix":         "cervix",
    "Colon":          "colon",
    "Esophagus":      "esophagus",
    "Heart":          "heart",
    "Kidney":         "kidney",
    "Liver":          "liver",
    "Lung":           "lung",
    "Muscle":         "muscle",
    "Nerve":          "nerve",
    "Ovary":          "ovary",
    "Pancreas":       "pancreas",
    "Pituitary":      "pituitary",
    "Prostate":       "prostate",
    "Skin":           "skin",
    "Small intestine":"small intestine",
    "Spleen":         "spleen",
    "Stomach":        "stomach",
    "Testis":         "testis",
    "Thyroid":        "thyroid",
    "Uterus":         "uterus",
    "Vagina":         "vagina",
    "Whole blood":    "whole blood",
}

### Build binary labels

For each tissue, positive samples are GSMs whose `source_name_ch1` or
`characteristics_ch1` contain the keyword and whose single-cell probability
is below the threshold.

In [ ]:
def get_binary_labels(keyword):
    """Return a binary Series (index = GSM in lv_matrix) for the given keyword."""
    kw   = keyword.lower()
    mask = (
        (np.char.find(np.char.lower(source), kw) >= 0) |
        (np.char.find(np.char.lower(char),   kw) >= 0)
    ) & (sc_prob < SC_PROB_THRESH)
    positive_gsms = set(geo_acc[mask].tolist()) & set(lv_matrix.index)
    y = pd.Series(
        [1 if gsm in positive_gsms else 0 for gsm in lv_matrix.index],
        index=lv_matrix.index,
        dtype=int,
    )
    return y


# Preview sample counts per tissue
counts = {t: get_binary_labels(kw).sum() for t, kw in TISSUE_KEYWORDS.items()}
counts_df = pd.DataFrame.from_dict(counts, orient="index", columns=["n_positive"])
counts_df["pass"] = counts_df["n_positive"] >= MIN_SAMPLES
print(counts_df.to_string())

## Binary RF + SHAP

For each tissue:
1. Build binary labels from keyword matching.
2. Nested 5×5 CV with `RandomizedSearchCV` for hyperparameter selection.
3. Train final model on all data and run SHAP `TreeExplainer`.
4. Collect positive-class mean SHAP values over tissue samples.

In [ ]:
_files_exist = (
    (OUTPUT_DIR / "accuracy_summary.tsv").exists() and
    (OUTPUT_DIR / "all_shap_positive.tsv").exists() and
    (OUTPUT_DIR / "cumulative_importance.tsv").exists()
)
if _files_exist:
    print("results found – skipping RF/SHAP computation")
else:
    print("results not found – running RF/SHAP computation")

In [ ]:
if not _files_exist:
    accuracy_list            = []
    shap_results_list        = []
    cumulative_importance_list = []

    (OUTPUT_DIR / "per_tissue").mkdir(parents=True, exist_ok=True)

    tissues = sorted(TISSUE_KEYWORDS.keys())

    for i, tissue in enumerate(tissues):
        keyword = TISSUE_KEYWORDS[tissue]
        y_binary = get_binary_labels(keyword)

        n_pos = int(y_binary.sum())
        n_neg = int((y_binary == 0).sum())

        if n_pos < MIN_SAMPLES:
            print(f"[{i+1}/{len(tissues)}] {tissue}: only {n_pos} positive samples – skipping")
            continue

        imbalance_ratio = n_neg / n_pos
        print(f"[{i+1}/{len(tissues)}] {tissue}  pos={n_pos:,}  neg={n_neg:,}  ratio={imbalance_ratio:.1f}")

        outer_cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED + i)
        outer_test_scores = []
        best_params_list  = []

        for outer_fold, (dev_idx, test_idx) in enumerate(outer_cv.split(lv_matrix, y_binary)):
            lv_dev  = lv_matrix.iloc[dev_idx]
            y_dev   = y_binary.iloc[dev_idx]
            lv_test = lv_matrix.iloc[test_idx]
            y_test  = y_binary.iloc[test_idx]

            inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED + i)
            clf = RandomForestClassifier(random_state=GLOBAL_SEED + i, class_weight="balanced")
            rs  = RandomizedSearchCV(
                estimator=clf,
                param_distributions=PARAM_GRID,
                n_iter=15,
                cv=inner_cv,
                scoring="balanced_accuracy",
                n_jobs=-1,
                random_state=GLOBAL_SEED + i,
            )
            rs.fit(lv_dev, y_dev)
            best_params_list.append(rs.best_params_)

            final_outer = RandomForestClassifier(
                random_state=GLOBAL_SEED + i,
                class_weight="balanced",
                **rs.best_params_,
            )
            final_outer.fit(lv_dev, y_dev)
            y_pred = final_outer.predict(lv_test)
            outer_test_scores.append(balanced_accuracy_score(y_test, y_pred))

        best_fold_idx = int(np.argmax(outer_test_scores))
        final_params  = best_params_list[best_fold_idx]
        mean_acc      = float(np.mean(outer_test_scores))
        std_acc       = float(np.std(outer_test_scores))
        print(f"  nested CV balanced accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

        # final model on all data
        final_model = RandomForestClassifier(
            random_state=GLOBAL_SEED + i,
            class_weight="balanced",
            **final_params,
        )
        final_model.fit(lv_matrix, y_binary)

        # SHAP – path-dependent TreeExplainer (no background data needed),
        # computed only on positive-class samples to avoid OOM
        tissue_mask = (y_binary == 1).values
        lv_tissue   = lv_matrix.iloc[tissue_mask]

        explainer        = shap.TreeExplainer(final_model)
        shap_explanation = explainer(lv_tissue)
        shap_class1      = shap_explanation.values[..., 1]   # positive class

        mean_shap_tissue = np.mean(shap_class1, axis=0)

        # free SHAP objects before next iteration
        del shap_explanation, shap_class1, explainer
        gc.collect()

        df_shap = pd.DataFrame({"Feature": lv_matrix.columns, "Mean_SHAP_Tissue": mean_shap_tissue})
        df_pos  = (
            df_shap[df_shap["Mean_SHAP_Tissue"] > 0]
            .sort_values("Mean_SHAP_Tissue", ascending=False)
            .reset_index(drop=True)
        )

        n_features_needed = {}
        if len(df_pos) > 0:
            total_pos_shap             = df_pos["Mean_SHAP_Tissue"].sum()
            df_pos["Cumulative_SHAP"]  = df_pos["Mean_SHAP_Tissue"].cumsum()
            df_pos["Cumulative_Pct"]   = df_pos["Cumulative_SHAP"] / total_pos_shap * 100
            df_pos["Rank"]             = range(1, len(df_pos) + 1)

            for thresh in [50, 70, 80, 90, 95]:
                n_features_needed[thresh] = int((df_pos["Cumulative_Pct"] >= thresh).idxmax()) + 1

            df_cum = df_pos[["Feature", "Mean_SHAP_Tissue", "Cumulative_Pct", "Rank"]].copy()
            df_cum["Tissue"] = tissue
            cumulative_importance_list.append(df_cum)

        accuracy_list.append({
            "Tissue":                      tissue,
            "N_Positive":                  n_pos,
            "Imbalance_Ratio":             imbalance_ratio,
            "Best_Test_Balanced_Accuracy": outer_test_scores[best_fold_idx],
            "Mean_CV_Accuracy":            mean_acc,
            "Std_CV_Accuracy":             std_acc,
            "N_Positive_LVs":              len(df_pos),
            "LVs_for_80pct":              n_features_needed.get(80),
            "LVs_for_90pct":              n_features_needed.get(90),
        })

        df_all_pos = df_pos.copy()
        df_all_pos["Tissue"] = tissue
        shap_results_list.append(df_all_pos)

        safe_name = tissue.replace(" ", "_").replace("/", "_")
        df_pos.to_csv(OUTPUT_DIR / "per_tissue" / f"shap_positive_{safe_name}.tsv", sep="\t", index=False)
        print(f"  top LVs: {df_pos['Feature'].head(5).tolist()}")

        del final_model, lv_tissue
        gc.collect()

    # compile
    accuracy_df   = pd.DataFrame(accuracy_list)
    shap_df       = pd.concat(shap_results_list,        ignore_index=True)
    cumulative_df = pd.concat(cumulative_importance_list, ignore_index=True)

    # filter by test accuracy
    passing = accuracy_df[accuracy_df["Best_Test_Balanced_Accuracy"] >= MIN_RF_ACCURACY]["Tissue"]
    accuracy_df   = accuracy_df[accuracy_df["Tissue"].isin(passing)]
    shap_df       = shap_df[shap_df["Tissue"].isin(passing)]
    cumulative_df = cumulative_df[cumulative_df["Tissue"].isin(passing)]
    print(f"\nTissues passing accuracy >= {MIN_RF_ACCURACY}: {sorted(passing.tolist())}")

    accuracy_df.to_csv(OUTPUT_DIR / "accuracy_summary.tsv",    sep="\t", index=False)
    shap_df.to_csv(    OUTPUT_DIR / "all_shap_positive.tsv",   sep="\t", index=False)
    cumulative_df.to_csv(OUTPUT_DIR / "cumulative_importance.tsv", sep="\t", index=False)
    print(f"\nDone. Results saved to: {OUTPUT_DIR}")

## Results

In [ ]:
accuracy_df   = pd.read_csv(OUTPUT_DIR / "accuracy_summary.tsv",     sep="\t")
shap_df       = pd.read_csv(OUTPUT_DIR / "all_shap_positive.tsv",    sep="\t")
cumulative_df = pd.read_csv(OUTPUT_DIR / "cumulative_importance.tsv", sep="\t")

In [ ]:
accuracy_df.sort_values("Mean_CV_Accuracy", ascending=False)

In [ ]:
cumulative_df.head(40)

In [ ]:
# Top 5 LVs per tissue
top5 = (
    shap_df
    .sort_values(["Tissue", "Mean_SHAP_Tissue"], ascending=[True, False])
    .groupby("Tissue")
    .head(5)
    .reset_index(drop=True)
)
top5